# Notebook 2 – Machine Learning Workflow

## Complete End-to-End Machine Learning Workflow

This notebook demonstrates a complete Machine Learning workflow using the Breast Cancer Wisconsin dataset.

## 1. Problem Definition

**Goal:** Predict whether a tumor is malignant or benign based on measured cell characteristics.

**Machine Learning Task:** Binary Classification

**Target Variable:** target

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Libraries imported successfully.')

## 2. Business Understanding

A classification model can help identify cases that may require further evaluation.

### Business Objectives
- Identify potentially malignant cases accurately.
- Reduce incorrect classifications.
- Compare different machine learning models.
- Select and tune the best model.

## 3. Data Collection

The dataset is collected from scikit-learn.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print('Dataset loaded successfully.')
print('Features:', X.shape)
print('Target:', y.shape)

## 4. Data Understanding

In [ ]:
print('First 5 rows:')
display(X.head())

print('\nDataset shape:', X.shape)
print('\nColumns:')
print(X.columns.tolist())

In [ ]:
X.info()

In [ ]:
print('Target distribution:')
print(y.value_counts())

print('\nTarget labels:')
for i, name in enumerate(data.target_names):
    print(i, '=', name)

In [ ]:
X.describe().T

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print('Total missing values:', X.isnull().sum().sum())
print('Duplicate rows:', X.duplicated().sum())

In [ ]:
target_counts = y.value_counts().sort_index()

plt.figure(figsize=(7,5))
plt.bar(data.target_names, target_counts.values)
plt.xlabel('Class')
plt.ylabel('Number of Samples')
plt.title('Target Class Distribution')
plt.show()

In [ ]:
selected_features = ['mean radius', 'mean texture', 'mean perimeter']

X[selected_features].hist(figsize=(10,4), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
correlation = X.corr()

plt.figure(figsize=(12,9))
plt.imshow(correlation, aspect='auto')
plt.colorbar(label='Correlation')
plt.title('Feature Correlation Matrix')
plt.xticks(range(len(X.columns)), X.columns, rotation=90, fontsize=6)
plt.yticks(range(len(X.columns)), X.columns, fontsize=6)
plt.tight_layout()
plt.show()

## 6. Data Preprocessing

The dataset is checked for missing values and duplicates. The data is then split into training and testing sets and features are standardized.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Feature scaling completed.')

## 7. Feature Engineering

Feature engineering creates new variables from existing variables.

In [ ]:
X_engineered = X.copy()

X_engineered['radius_texture_ratio'] = (
    X_engineered['mean radius'] / (X_engineered['mean texture'] + 1e-8)
)

display(X_engineered[['mean radius', 'mean texture', 'radius_texture_ratio']].head())

## 8. Train/Test Split

80% of the data is used for training and 20% for testing.

The test data remains unseen during model training.

## 9. Model Selection

We will compare:
- Logistic Regression
- K-Nearest Neighbors
- Support Vector Machine
- Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Support Vector Machine': SVC(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42)
}

print('Models selected:')
for name in models:
    print('-', name)

## 10. Model Training

In [ ]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(name, 'trained successfully.')

## 11. Prediction

In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test_scaled)

print('Predictions generated.')

## 12. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name in trained_models:
    y_pred = predictions[name]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1 Score', ascending=False)

display(results_df)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_initial_model_name = results_df.iloc[0]['Model']
best_initial_predictions = predictions[best_initial_model_name]

print('Best initial model:', best_initial_model_name)
print()
print(classification_report(
    y_test,
    best_initial_predictions,
    target_names=data.target_names
))

In [ ]:
cm = confusion_matrix(y_test, best_initial_predictions)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.colorbar()
plt.xticks([0,1], data.target_names)
plt.yticks([0,1], data.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha='center', va='center')

plt.show()

## 13. Hyperparameter Tuning

GridSearchCV is used to find the best parameters for the Support Vector Machine.

In [ ]:
from sklearn.model_selection import GridSearchCV

svm = SVC(random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'kernel': ['linear', 'rbf']
}

grid_search = GridSearchCV(
    svm,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print('Best parameters:')
print(grid_search.best_params_)

print('\nBest CV F1 Score:')
print(grid_search.best_score_)

## 14. Final Model Selection

In [ ]:
best_model = grid_search.best_estimator_
y_pred_final = best_model.predict(X_test_scaled)

final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)

print('Final Model: Tuned SVM')
print('Accuracy :', final_accuracy)
print('Precision:', final_precision)
print('Recall   :', final_recall)
print('F1 Score :', final_f1)

print('\nClassification Report:')
print(classification_report(
    y_test,
    y_pred_final,
    target_names=data.target_names
))

## 15. Deployment

The final model and scaler can be saved using joblib and loaded later by an application.

In [ ]:
import joblib

joblib.dump(best_model, 'breast_cancer_model.pkl')
joblib.dump(scaler, 'breast_cancer_scaler.pkl')

print('Model saved as breast_cancer_model.pkl')
print('Scaler saved as breast_cancer_scaler.pkl')

In [ ]:
# Load the saved model and scaler
loaded_model = joblib.load('breast_cancer_model.pkl')
loaded_scaler = joblib.load('breast_cancer_scaler.pkl')

# Example prediction
sample = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)

prediction = loaded_model.predict(sample_scaled)[0]

print('Predicted class:', data.target_names[prediction])
print('Actual class:', data.target_names[y_test.iloc[0]])

# Complete ML Workflow

```text
Problem Definition
        ↓
Business Understanding
        ↓
Data Collection
        ↓
Data Understanding
        ↓
EDA
        ↓
Data Preprocessing
        ↓
Feature Engineering
        ↓
Train/Test Split
        ↓
Model Selection
        ↓
Model Training
        ↓
Prediction
        ↓
Evaluation
        ↓
Hyperparameter Tuning
        ↓
Final Model Selection
        ↓
Deployment
```

## Conclusion

This notebook demonstrates an end-to-end Machine Learning workflow from defining the problem to deploying a trained model.